![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/visual-nlp-workshop/blob/master/jupyter/deid-and-obfuscation/SparkOCRBulkOperation.ipynb)

# PDF Bulk De-Identification Pipeline (Visual NLP)

> ⚠️ **Runs on Databricks only.**

**What this notebook does:** takes PDF documents and blacks out private patient information (PHI) printed on the pages, then writes the de-identified PDFs back to a folder.

**The workflow in plain steps**

| # | Step | What happens |
|---|------|--------------|
| 1 | Read PDF files | Load the files from a folder |
| 2 | OCR | Read the text on each page (Tesseract, text detector + OCR, *or* a vision-language model) |
| 3 | NLP | Find the sensitive words (names, dates, IDs, contacts) in that text |
| 4 | Coordinates | Work out where each sensitive word is on the page |
| 5 | De-identify | Black out those areas and rebuild the PDF |
| 6 | Check | Count the output files and look at the result |

## Cluster Configuration (Databricks)

Create a Databricks cluster with these settings (**Compute → your cluster → Advanced options → Spark config**), and restart it after any change.

| Setting | CPU cluster | GPU cluster |
|---|---|---|
| Workers | `i4i.2xlarge` × 4 (fixed, min 4 / max 4) | `g5.2xlarge` (A10G) × 4 (fixed, min 4 / max 4) |
| Driver | `i4i.2xlarge` | `g5.2xlarge` (A10G) |
| Photon acceleration | Disabled | Disabled |
| `spark.sql.legacy.allowUntypedScalaUDF` | `true` | `true` |
| `spark.executor.memoryOverhead` | `16g` | `16g` |
| `spark.task.cpus` | `4` | default |

**CPU concurrency:** each `i4i.2xlarge` has 8 vCPUs. With `spark.task.cpus = 4`, each executor runs 8 / 4 = **2 tasks at a time**, so the cluster runs at most 4 × 2 = **8 concurrent tasks**.

**Tip:** Spark is *lazy* - many cells return instantly because they only build a plan. The real work happens when data is shown, counted or written.

## Import transformers and annotators

In [0]:
import os
import json 
import sys
import shutil 

from sparknlp.annotator import *
from sparknlp.base import *
import sparknlp_jsl
from sparknlp_jsl.annotator import *

import sparkocr
from sparkocr.transformers import *
from sparkocr.utils import *
from sparkocr.enums import *
from sparkocr.schemas import BinarySchema

from pyspark.ml import PipelineModel, Pipeline
import pyspark.sql.functions as F

print(f"Spark NLP version: {sparknlp.version()}")
print(f"Spark NLP internal version: {sparknlp_jsl.version()}")
print(f"Spark OCR version: {sparkocr.version()}")

Spark NLP version: 6.4.2
Spark NLP internal version: 6.4.1
Spark OCR version: 6.4.3rc10


## Define the pipelines
The next cells only **define** functions that build pipelines; nothing runs until they are called further down.

- **NLP pipeline** - finds sensitive text and its coordinates
- **Tesseract OCR pipeline** - CPU OCR
- **Text detector + OCR pipeline** - detects text regions, then reads them (detector can use GPU)
- **VLM OCR pipeline** - a vision-language model reads text and coordinates [ Requires a GPU ]
- **Finalizer** - blacks out the text and rebuilds the PDF

In [0]:
def nlp_deid_pipeline():
    
    document_assembler = DocumentAssembler() \
        .setInputCol("text") \
        .setOutputCol("document")

    sentence_detector = SentenceDetector() \
        .setInputCols(["document"]) \
        .setOutputCol("sentence")
    
    labels = ["NAME", "DATE", "ID", "CONTACT"]
    zeroshot_ner_deid_generic_nonMedical_medium = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_nonMedical_medium", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_nonMedical_medium")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ["IDNUM","MEDICALRECORD", "NAME","PATIENT", "PHONE"]
    zeroshot_ner_deid_subentity_nonMedical_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_subentity_nonMedical_large", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_subentity_nonMedical_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ['CONTACT', 'DATE', 'ID', 'NAME'] 
    zeroshot_ner_deid_generic_multi_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_multi_large", "xx", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_multi_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)
    
    date_regex_matcher = RegexMatcherInternalModel.pretrained("date_matcher","en","clinical/models") \
        .setInputCols(["sentence"]) \
        .setOutputCol("date_chunk")
    
    chunk_merger = ChunkMergeApproach()\
        .setInputCols("ner_zeroshot_ner_deid_generic_nonMedical_medium", "ner_zeroshot_ner_deid_subentity_nonMedical_large","ner_zeroshot_ner_deid_generic_multi_large", "date_chunk")\
        .setOutputCol("merged_chunk")\
        .setMergeOverlapping(True)

    position_finder = PositionFinder() \
        .setInputCols("merged_chunk") \
        .setOutputCol("coordinates") \
        .setSplitOnNewLines(True) \
        .setPageMatrixCol("positions") \
        .setIgnoreSchema(True)

    nlp_pipeline = Pipeline(stages=[
        document_assembler,
        sentence_detector,
        zeroshot_ner_deid_generic_nonMedical_medium,
        zeroshot_ner_deid_subentity_nonMedical_large,
        zeroshot_ner_deid_generic_multi_large,
        date_regex_matcher,
        chunk_merger,
        position_finder
    ])

    empty_data = spark.createDataFrame([[""]]).toDF("text")
    nlp_model = nlp_pipeline.fit(empty_data)
    return nlp_model

In [0]:
def ocr_tesseract_pipeline_builder(resolution=300):
    
    pdf_to_image = PdfToImage() \
        .setInputCol("content") \
        .setOutputCol("image") \
        .setResolution(resolution) \
        .setKeepInput(False) \
        .setCompressImage(True) \
        .setImageDimsCol("frame_dims")
    
    ocr = ImageToText() \
        .setInputCol("image") \
        .setOutputCol("text") \
        .setIgnoreResolution(False) \
        .setPageIteratorLevel(PageIteratorLevel.SYMBOL) \
        .setPageSegMode(PageSegmentationMode.SPARSE_TEXT) \
        .setWithSpaces(True) \
        .setKeepLayout(False) \
        .setConfidenceThreshold(70)
        
    ocr_pipeline = PipelineModel(stages=[
        pdf_to_image,
        ocr
    ])

    return ocr_pipeline

In [0]:
def ocr_v3_pipeline_builder(resolution=300, use_gpu=False):
    
    pdf_to_image = PdfToImage() \
        .setInputCol("content") \
        .setOutputCol("image") \
        .setResolution(resolution) \
        .setKeepInput(False) \
        .setCompressImage(True) \
        .setImageDimsCol("frame_dims")
    
    text_detector = ImageTextDetector.pretrained("image_text_detector_mem_opt", "en", "clinical/ocr") \
        .setInputCol("image") \
        .setOutputCol("text_regions") \
        .setWithRefiner(True) \
        .setUseGPU(use_gpu) \
        .setWidth(0)
    
    ocr = ImageToTextV3() \
        .setInputCols(["image", "text_regions"]) \
        .setOutputCol("text")
        
    ocr_pipeline = PipelineModel(stages=[
        pdf_to_image,
        text_detector,
        ocr
    ])

    return ocr_pipeline

In [0]:
def ocr_vlm_pipeline_builder(resolution=300):
    
    pdf_to_image = PdfToImage() \
        .setInputCol("content") \
        .setOutputCol("image") \
        .setResolution(resolution) \
        .setKeepInput(False) \
        .setCompressImage(True) \
        .setImageDimsCol("frame_dims")
    
    caption_assembler = DocumentAssembler() \
        .setInputCol("caption") \
        .setOutputCol("caption_document")

    schema_converter_assembler = ImageSchemaConverter() \
        .setInputCol("image") \
        .setOutputCol("image_assembler") \
        .setOutputSchema(ImageSchemaConversion.ASSEMBLER) \
        .setKeepInput(False)

    ocr = MedicalVisionLLM.pretrained("jsl-ocr-gguf-vlm1", "en", "clinical/ocr") \
        .setInputCols(["caption_document", "image_assembler"]) \
        .setOutputCol("completions") \
        .setNGpuLayers(99) \
        .setNCtx(32768) \
        .setNParallel(1) \
        .setNBatch(2048) \
        .setNUbatch(1024) \
        .setNPredict(1024) \
        .setTemperature(0.01) \
        .setTopK(1) \
        .setTopP(1.0) \
        .setRepeatPenalty(1.03) \
        .setRepeatLastN(256) \
        .setStopStrings(["<\uff5chy_Assistant\uff5c>", "<\uff5chy_place\u2581holder\u2581no\u25812\uff5c>"]) \
        .setMinKeep(0) \
        .setNProbs(0) \
        .setBatchSize(1) \
        .setDisableLog(False)

    coordinate_extract = DocumentCoordinatesToText() \
        .setInputCol("completions") \
        .setImageDimsCol("frame_dims") \
        .setOutputCol("text") \
        .setPageMatrixCol("positions") \
        .setRegionCol("regions")
    
    vlm_ocr_pipeline = PipelineModel(stages=[
        pdf_to_image,
        caption_assembler,
        schema_converter_assembler,
        ocr,
        coordinate_extract
    ])

    return vlm_ocr_pipeline

In [0]:
def pdf_finalizer_stage(resolution=300):

    draw_regions = ImageDrawRegions() \
        .setInputCol("image") \
        .setInputRegionsCol("coordinates") \
        .setRectColor(Color.black) \
        .setFilledRect(True) \
        .setOutputCol("image_with_regions")

    img_to_pdf = ImageToPdf() \
        .setInputCol("image_with_regions") \
        .setPageNumCol("pagenum") \
        .setOriginCol("path") \
        .setOutputCol("pdf") \
        .setAggregatePages(True)

    pdf_finisher_pipe = PipelineModel(stages=[
        draw_regions,
        img_to_pdf
    ])

    return pdf_finisher_pipe

## Settings
Change the values in the next cell to fit your data. A summary printed afterwards lets you double-check them.

- resolution: DPI used to render PDF pages to images for OCR. Higher = better reading of small text but slower and more memory.
- useGPU: True if the cluster has a GPU. Set to False for CPU-only clusters.

In [0]:
# PDF IMAGE SETTING
resolution = 300

#GPU SETTING
useGPU = True 

# I/O SETTING
input_load_path = "/Volumes/visual_nlp/default/visual_nlp/pdfs/input"
output_save_path = "/Volumes/visual_nlp/default/visual_nlp/pdfs/output"

ocr_save_path = os.path.join(output_save_path, "ocr")
nlp_save_path = os.path.join(output_save_path, "nlp")

In [0]:
# ---------------------------------------------------------------
# CONFIG SUMMARY
# ---------------------------------------------------------------
def _section(title, rows, width=22):
    print(f"\n  {title}")
    print("  " + "─" * 70)
    for key, value in rows:
        print(f"  {key:<{width}} {value}")

print("═" * 74)
print("PDF DE-IDENTIFICATION PIPELINE · CONFIGURATION")
print("═" * 74)

_section("🖼PDF IMAGE SETTINGS", [
    ("resolution",  resolution)
])

_section("⚡ GPU SETTINGS", [
    ("useGPU", f"{'ENABLED ✅' if useGPU else 'DISABLED ❌ (CPU only)'}"),
])

_section("📂 INPUT", [
    ("input_load_path", input_load_path),
])

_section("💾 OUTPUT", [
    ("output_save_path", output_save_path),
    ("ocr_save_path", ocr_save_path),
    ("nlp_save_path", nlp_save_path),
], width=26)

print("\n" + "═" * 74)

══════════════════════════════════════════════════════════════════════════
PDF DE-IDENTIFICATION PIPELINE · CONFIGURATION
══════════════════════════════════════════════════════════════════════════

  🖼PDF IMAGE SETTINGS
  ──────────────────────────────────────────────────────────────────────
  resolution             300

  ⚡ GPU SETTINGS
  ──────────────────────────────────────────────────────────────────────
  useGPU                 ENABLED ✅

  📂 INPUT
  ──────────────────────────────────────────────────────────────────────
  input_load_path        /Volumes/visual_nlp/default/visual_nlp/pdfs/input

  💾 OUTPUT
  ──────────────────────────────────────────────────────────────────────
  output_save_path           /Volumes/visual_nlp/default/visual_nlp/pdfs/output
  ocr_save_path              /Volumes/visual_nlp/default/visual_nlp/pdfs/output/ocr
  nlp_save_path              /Volumes/visual_nlp/default/visual_nlp/pdfs/output/nlp

═══════════════════════════════════════════════════════════

## Define NLP De-Identification Pipeline

In [0]:
ner_coordinate_pipeline = nlp_deid_pipeline()

## Choose ONE OCR Method

> **Run only one of Option A, B or C.** They share the names `ocr_model` and `ocr_result`, so the next option overwrites the previous one.

| | **Option A: ImageToText** | **Option B: Detector + ImageToTextV3** | **Option C: VLM OCR** |
|---|---|---|---|
| How it works | Tesseract reads the page | Detects text regions, then reads them | One vision-language model reads text and coordinates |
| GPU | No | Optional (detector) | Required |

Run all cells of your chosen section, skip the others, then continue with **Dump OCR Results To Disk**.

### Option A: ImageToText (CPU)

In [0]:
ocr_model = ocr_tesseract_pipeline_builder(resolution=resolution)

In [0]:
# Load PDF FILES

df = spark.read.format("binaryFile").load(input_load_path) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))   
     
print(f"Total Files in Batch: {df.count()}")

Total Files in Batch: 110


In [0]:
ocr_result = ocr_model.transform(df).drop("image")
ocr_result.columns

['path',
 'modificationTime',
 'length',
 'total_pages',
 'pagenum',
 'documentnum',
 'frame_dims',
 'confidence',
 'exception',
 'text',
 'positions']

### Option B: ImageTextDetector (GPU) + ImageToTextV3 (CPU)

In [0]:
ocr_model = ocr_v3_pipeline_builder(resolution=resolution, use_gpu=useGPU)

image_text_detector_mem_opt download started this may take some time.
Approximate size to download 77.5 MB


In [0]:
# Load PDF FILES

df = spark.read.format("binaryFile").load(input_load_path) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))   
     
print(f"Total Files in Batch: {df.count()}")

Total Files in Batch: 110


In [0]:
ocr_result = ocr_model.transform(df).drop("image")
ocr_result.columns

['path',
 'modificationTime',
 'length',
 'total_pages',
 'pagenum',
 'documentnum',
 'frame_dims',
 'text_regions',
 'text',
 'positions',
 'exception']

### Option C: VLM Based OCR Text Extractor (GPU)

In [0]:
ocr_model = ocr_vlm_pipeline_builder(resolution=resolution)

jsl-ocr-gguf-vlm1 download started this may take some time.
Approximate size to download 1.5 GB


In [0]:
# Load PDF FILES

vision_prompt = "Detect and recognize text in the image, and output the text coordinates in a formatted manner."

df = spark.read.format("binaryFile").load(input_load_path) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", "")) \
    .withColumn("caption", F.lit(vision_prompt))
    
print(f"Total Files in Batch: {df.count()}")

Total Files in Batch: 110


In [0]:
ocr_result = ocr_model.transform(df).drop("image_assembler")
ocr_result.columns

['path',
 'modificationTime',
 'length',
 'caption',
 'total_pages',
 'exception',
 'pagenum',
 'documentnum',
 'frame_dims',
 'caption_document',
 'completions',
 'text',
 'regions',
 'positions']

## Dump OCR Results To Disk

In [0]:
ocr_result.write.format("parquet").mode("overwrite").save(ocr_save_path)

## Generate NLP Result

In [0]:
ocr_result_df_from_disk = spark.read.format("parquet").load(ocr_save_path)

ocr_result_df_from_disk.select("exception").distinct().show(100, False)

+---------+
|exception|
+---------+
|         |
+---------+



In [0]:
nlp_result = ner_coordinate_pipeline.transform(ocr_result_df_from_disk)
nlp_result.columns

['path',
 'modificationTime',
 'length',
 'total_pages',
 'pagenum',
 'documentnum',
 'frame_dims',
 'confidence',
 'exception',
 'text',
 'positions',
 'document',
 'sentence',
 'ner_zeroshot_ner_deid_generic_nonMedical_medium',
 'ner_zeroshot_ner_deid_subentity_nonMedical_large',
 'ner_zeroshot_ner_deid_generic_multi_large',
 'date_chunk',
 'merged_chunk',
 'coordinates']

In [0]:
nlp_result.write.format("parquet").mode("overwrite").save(nlp_save_path)

## Generate De-Id Result

In [0]:
nlp_result_df_from_disk = spark.read.format("parquet").load(nlp_save_path)

nlp_result_df_from_disk.select("exception").distinct().show(100, False)

+---------+
|exception|
+---------+
|         |
+---------+



## Render Redactions and Rebuild PDFs
Reload the page images, join them with the NLP coordinates (per file and page), black out the PHI regions and merge the pages back into PDFs.

In [0]:
df = spark.read.format("binaryFile").load(input_load_path) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

In [0]:
pdf_to_image = PdfToImage() \
    .setInputCol("content") \
    .setOutputCol("image") \
    .setResolution(resolution) \
    .setKeepInput(False) \
    .setCompressImage(True) \
    .setImageDimsCol("frame_dims")

input_image_df = pdf_to_image.transform(df).select("path", "image", "pagenum") \
    .withColumnRenamed("path", "second_pass_path") \
    .withColumnRenamed("pagenum", "second_pass_pagenum")

input_image_df.columns

['second_pass_path', 'image', 'second_pass_pagenum']

In [0]:
joined_df = input_image_df.join(
    nlp_result_df_from_disk,
    on=((input_image_df["second_pass_path"] == nlp_result_df_from_disk["path"]) &
        (input_image_df["second_pass_pagenum"] == nlp_result_df_from_disk["pagenum"])),
    how="inner"
).drop("second_pass_pagenum", "second_pass_path")

joined_df.columns

['image',
 'path',
 'modificationTime',
 'length',
 'total_pages',
 'pagenum',
 'documentnum',
 'frame_dims',
 'confidence',
 'exception',
 'text',
 'positions',
 'document',
 'sentence',
 'ner_zeroshot_ner_deid_generic_nonMedical_medium',
 'ner_zeroshot_ner_deid_subentity_nonMedical_large',
 'ner_zeroshot_ner_deid_generic_multi_large',
 'date_chunk',
 'merged_chunk',
 'coordinates']

In [0]:
pdf_pipeline = pdf_finalizer_stage(resolution=resolution)

In [0]:
final_result = pdf_pipeline.transform(joined_df)
final_result.columns

['path', 'images', 'pages', 'pdf', 'exception']

## Write De-Identified PDFs

Saves the final results in one pass:

1. **Cleaned PDF files** are written to `<output_save_path>/pdf/`, in parallel by each worker, using the original file name.
2. **A status table** is written as Parquet to `<output_save_path>/metadata/` with the columns `path` and `exception` (error message if the file failed, otherwise null).

In [0]:
import os
import pyarrow as pa
from pyspark.sql.types import StructType

pdf_bytes = "pdf"
exception_col = "exception"
path_col = "path"

meta_cols = [path_col, exception_col]
out_schema = StructType([final_result.schema[c] for c in meta_cols])

# Path for Output PDF Bytes
pdf_root = os.path.join(output_save_path, "pdf")
os.makedirs(pdf_root, exist_ok=True)

# Path for Output file path / Exception table
parquet_path = os.path.join(output_save_path, "metadata")
os.makedirs(parquet_path, exist_ok=True)

def write_pdfs(batches):
    import os
    import pyarrow as pa
    for batch in batches:
        tbl = pa.Table.from_batches([batch])
        for p, b in zip(tbl.column("path").to_pylist(), tbl.column(pdf_bytes).to_pylist()):
            with open(os.path.join(pdf_root, os.path.basename(p)), "wb") as f:
                f.write(b)
        yield from tbl.select(meta_cols).to_batches()

# Using overwrite for this example. Change the write mode to append if running in batches,
# so that previous results are not overridden (or write each batch to a different folder).
final_result.select(path_col, pdf_bytes, exception_col) \
    .mapInArrow(write_pdfs, out_schema) \
    .write.mode("overwrite") \
    .parquet(parquet_path)

## Check Result

In [0]:
df = spark.read.format("parquet").load(parquet_path)

df.select("exception").distinct().show(100, False)

+---------+
|exception|
+---------+
|         |
+---------+



In [0]:
print(f'Total De-Identified Files : {df.select("path").distinct().count()}')

Total De-Identified Files : 110


In [0]:
df = spark.read.format("binaryFile").load(pdf_root).withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

print(f"Total Deid Files: {df.count()}")

Total Deid Files: 110


In [0]:
display_pdf(df=df.limit(1), field="content",  width=700, show_meta=True)